# Week 1 — The Agentic Loop, by hand

*Course project: a personal assistant that helps a busy student organise their life.*

**Goal for today:** build a working agent from scratch with nothing but the Anthropic SDK and a `while` loop.

> An agent is a model in a loop. On each turn the model either **answers** or **asks for a tool**.
> We run the tool, append the result to the conversation, and call the model again.

By the end of this notebook you will have:
1. Seen why a single LLM call is not enough
2. Done one tool round-trip **by hand** (you are the loop)
3. Written the loop
4. Broken it on purpose and fixed it
5. Learned to read an agent trace
6. Compared your loop with real-world agents (Hermes, OpenClaw)

**No frameworks today.** Every line that talks to the model is code you can read.

## 0. Setup

```bash
pip install -r requirements.txt
export ANTHROPIC_API_KEY=sk-ant-...      # or: ant auth login
```

In [ ]:
import json
from pathlib import Path

import anthropic

client = anthropic.Anthropic()          # reads ANTHROPIC_API_KEY (or your `ant auth login` profile)
MODEL = "claude-opus-5-5"
DATA_DIR = Path("data").resolve()

print("SDK version:", anthropic.__version__)
print("Data files:", sorted(p.name for p in DATA_DIR.iterdir()))

## 1. One LLM call is not an agent

Ask the model something it **cannot know** (our private files) and something it **should not guess** (exact arithmetic).

In [ ]:
response = client.messages.create(
    model=MODEL,
    max_tokens=4000,
    messages=[{"role": "user", "content":
        "What's on my calendar on Tuesday 13 October (data/calendar.csv)? Also, what is 1234.56 * 987.65 + 0.07 ** 3 exactly?"}],
)
print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "text":
        print(block.text)

**Discuss:** What did the model do with the file question? Check the arithmetic in Python (`1234.56 * 987.65 + 0.07 ** 3`). Was it exact?

The model has no eyes and no calculator. **Tools** give it both.

## 2. One tool, one round-trip — *you* are the loop

A tool has two halves:

| Half | Who uses it | What it is |
|---|---|---|
| **Schema** | the model | name + description + JSON schema of the arguments |
| **Function** | your code | the Python that actually runs |

The model never executes anything. It only **asks**.

In [ ]:
calculator_schema = {
    "name": "calculator",
    "description": "Evaluate an arithmetic expression exactly. Supports + - * / ** % and parentheses.",
    "input_schema": {
        "type": "object",
        "properties": {"expression": {"type": "string", "description": "e.g. '(2 + 3) * 4'"}},
        "required": ["expression"],
    },
}

messages = [{"role": "user", "content": "What is 1234.56 * 987.65 + 0.07 ** 3? Use the calculator."}]

response = client.messages.create(
    model=MODEL, max_tokens=4000, tools=[calculator_schema], messages=messages,
)
print("stop_reason:", response.stop_reason)       # <- expect "tool_use"
for block in response.content:
    print(block.type, "->", block.model_dump(exclude={"signature"}))

`stop_reason == "tool_use"` means: *"I'm not done — please run this and tell me what happened."*

Let's write the function. **Never use `eval()` on model output** — that would let the model run any Python on your machine. We walk the syntax tree and only allow arithmetic.

In [ ]:
import ast, operator

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.Pow: operator.pow, ast.Mod: operator.mod,
        ast.USub: operator.neg, ast.UAdd: operator.pos}

def _eval_node(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.left), _eval_node(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_eval_node(node.operand))
    raise ValueError(f"Unsupported expression element: {ast.dump(node)}")

def calculator(expression: str) -> str:
    return str(round(_eval_node(ast.parse(expression, mode="eval").body), 6))

print(calculator("1234.56 * 987.65 + 0.07 ** 3"))

Now do by hand what the loop will do automatically:

1. Append the assistant's **whole** reply (all blocks, not just text) to `messages`
2. Run the tool
3. Send back a `tool_result` whose `tool_use_id` matches the request
4. Call the model again

In [ ]:
messages.append({"role": "assistant", "content": response.content})          # 1

tool_use = next(b for b in response.content if b.type == "tool_use")
output = calculator(**tool_use.input)                                        # 2

messages.append({"role": "user", "content": [                                # 3
    {"type": "tool_result", "tool_use_id": tool_use.id, "content": output}
]})

response = client.messages.create(                                           # 4
    model=MODEL, max_tokens=4000, tools=[calculator_schema], messages=messages,
)
print("stop_reason:", response.stop_reason)       # <- expect "end_turn"
print(next(b.text for b in response.content if b.type == "text"))

**Look at the shape of `messages`:**

In [ ]:
for m in messages:
    kinds = [c["type"] if isinstance(c, dict) else c.type for c in m["content"]] if isinstance(m["content"], list) else ["text"]
    print(f"{m['role']:9} {kinds}")

Notice: tool results travel in a **`user`** message. From the API's point of view, *the environment* speaks as the user.

**Question:** What if the model needed 7 tool calls instead of 1? → We need a loop.

## 3. A tool registry

To loop over *any* tool we need a lookup: `name → (schema, function)`.
Let's add two file tools. They are **sandboxed**: the model can only see inside `data/`.

In [ ]:
def _safe_path(path: str) -> Path:
    target = (DATA_DIR / path).resolve()
    if not target.is_relative_to(DATA_DIR):
        raise PermissionError(f"'{path}' is outside the data folder")
    return target

def list_dir(path: str = ".") -> str:
    target = _safe_path(path)
    if not target.is_dir():
        raise FileNotFoundError(f"No such directory: '{path}'")
    return "\n".join(f"{p.name}/" if p.is_dir() else p.name for p in sorted(target.iterdir())) or "(empty)"

def read_file(path: str) -> str:
    target = _safe_path(path)
    if not target.is_file():
        raise FileNotFoundError(f"No such file: '{path}'. Use list_dir to see what exists.")
    return target.read_text(encoding="utf-8")[:20_000]

TOOLS = {
    "list_dir": {"fn": list_dir, "schema": {
        "name": "list_dir",
        "description": "List files and subfolders in the data folder. Folders end with '/'. Call this first when you don't know which files exist.",
        "input_schema": {"type": "object", "properties": {
            "path": {"type": "string", "description": "Folder relative to the data folder. Use '.' for the root."}},
            "required": []}}},
    "read_file": {"fn": read_file, "schema": {
        "name": "read_file",
        "description": "Read a UTF-8 text file (e.g. CSV, TXT) from the data folder and return its full contents.",
        "input_schema": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File path relative to the data folder, e.g. 'calendar.csv'."}},
            "required": ["path"]}}},
    "calculator": {"fn": calculator, "schema": {
        **calculator_schema,
        "description": calculator_schema["description"] + " Put a whole sum in one expression, e.g. '120*19.99 + 45*49.5'."}},
}

print(list_dir())

`execute_tool` turns one `tool_use` block into one `tool_result` block.

**Key design decision:** errors are **returned to the model**, not raised. A crashed agent learns nothing; a model that reads `FileNotFoundError: ... Use list_dir` can fix its own mistake.

In [ ]:
def execute_tool(block, tools=TOOLS) -> dict:
    result = {"type": "tool_result", "tool_use_id": block.id}
    tool = tools.get(block.name)
    if tool is None:
        result.update(content=f"Unknown tool '{block.name}'. Available: {', '.join(tools)}", is_error=True)
        return result
    try:
        result["content"] = tool["fn"](**block.input)
    except Exception as exc:
        result.update(content=f"{type(exc).__name__}: {exc}", is_error=True)
    return result

# Try it with a fake block — no API call needed
from types import SimpleNamespace
print(execute_tool(SimpleNamespace(id="test", name="read_file", input={"path": "nope.csv"})))
print(execute_tool(SimpleNamespace(id="test", name="read_file", input={"path": "../../etc/passwd"})))

## 4. Close the loop

Every `stop_reason` is a branch of the loop:

| `stop_reason` | Meaning | What the loop does |
|---|---|---|
| `tool_use` | "run these tools for me" | execute all, append results, **continue** |
| `end_turn` | "I'm done" | return the answer |
| `max_tokens` | reply was cut off | stop — don't run tool calls with possibly truncated arguments |
| `refusal` | the model declined | stop and report |

Plus one condition **we** own: `max_turns`. The model decides *when* it is done; we decide *how long we're willing to wait*.

Two small production details in the call below:
- `output_config={"effort": ...}` — how hard the model thinks per turn (`low` … `max`).
- `fallbacks="default"` (beta) — if a safety classifier declines, the API retries on a fallback model for us.

In [ ]:
SYSTEM_PROMPT = (
    "You are a personal assistant for a busy university student. Today is Monday 2026-10-12. "
    "Their calendar, task list and notes are files in a sandboxed data folder. Never do arithmetic "
    "in your head: use the calculator. When you have the answer, reply briefly and show the key "
    "numbers you used."
)

def _short(x, n=200):
    s = str(x).replace("\n", " | ")
    return s if len(s) <= n else s[:n] + "..."

def trace_response(turn, response):
    u = response.usage
    print(f"\n── turn {turn} · stop_reason={response.stop_reason} · in={u.input_tokens} out={u.output_tokens} tokens")
    for b in response.content:
        if b.type == "text" and b.text.strip():
            print("  💬", _short(b.text, 300))
        elif b.type == "tool_use":
            print(f"  🔧 {b.name}({b.input})")

def run_agent(task, tools=TOOLS, max_turns=10, effort="medium", verbose=True):
    messages = [{"role": "user", "content": task}]

    for turn in range(1, max_turns + 1):
        response = client.beta.messages.create(
            model=MODEL,
            max_tokens=16000,
            system=SYSTEM_PROMPT,
            tools=[t["schema"] for t in tools.values()],
            messages=messages,
            output_config={"effort": effort},
            betas=["server-side-fallback-2026-07-01"],
            fallbacks="default",
        )
        if verbose:
            trace_response(turn, response)

        # Append the WHOLE reply. History is append-only: never edit earlier messages.
        messages.append({"role": "assistant", "content": response.content})

        if response.stop_reason == "tool_use":
            results = [execute_tool(b, tools) for b in response.content if b.type == "tool_use"]
            if verbose:
                for r in results:
                    print("  ❌" if r.get("is_error") else "  ✅", _short(r["content"]))
            messages.append({"role": "user", "content": results})   # ALL results in ONE message
            continue

        text = "\n".join(b.text for b in response.content if b.type == "text").strip()
        if response.stop_reason == "max_tokens":
            text += "\n[cut off: max_tokens]"
        elif response.stop_reason == "refusal":
            text = "[the model declined this request]"
        return text, messages

    return f"[stopped after {max_turns} turns without a final answer]", messages

In [ ]:
TASK = ("Which open tasks are due before my ML midterm on Friday, how many hours of work are they in "
        "total, and how much free time do I have between 09:00 and 18:00 from Monday to Thursday? "
        "Is it enough?")

answer, history = run_agent(TASK)
print("\n" + "═" * 60 + "\n" + answer)

**Never trust, verify.** Here is the ground truth computed with plain Python. Does the agent agree?

In [ ]:
import csv

def hours(hhmm):
    h, m = map(int, hhmm.split(":"))
    return h + m / 60

events = list(csv.DictReader(open(DATA_DIR / "calendar.csv")))
tasks = list(csv.DictReader(open(DATA_DIR / "tasks.csv")))

due = [t for t in tasks if t["status"] == "todo" and t["due"] < "2026-10-16"]
need = sum(float(t["est_hours"]) for t in due)
print("Due before Friday:", ", ".join(t["title"] for t in due), f"→ {need} h of work")

total_free = 0
for day in ("2026-10-12", "2026-10-13", "2026-10-14", "2026-10-15"):
    # clip events to the 09:00-18:00 window, then MERGE overlapping ones
    busy = sorted((max(hours(e["start"]), 9), min(hours(e["end"]), 18)) for e in events if e["date"] == day)
    merged = []
    for start, end in (b for b in busy if b[1] > b[0]):
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    free = 9 - sum(end - start for start, end in merged)
    total_free += free
    print(f"{day}: {free} h free")
print(f"Free time Mon-Thu: {total_free} h  vs  {need} h needed  →  margin {total_free - need} h")

**Discuss:**
- How many turns did it take? Did it call several tools in **one** turn (parallel tool use)?
- Tuesday has two **overlapping** events (study group 14:00–15:00, project sync 14:30–15:30). If the agent just
  added up event durations, its free time is off by 0.5 h. Did it notice?
- Did it read `notes/preferences.txt` or the syllabus? Should it have?
- "Is it enough?" A 2-hour margin across four days, with a café shift on Wednesday. Who decides what "enough" means?
  (Ambiguity in the *task* is not a loop bug.)

## 5. Break it on purpose

Agents fail in ways normal programs don't. Run each experiment, then write down **what you expected** vs **what happened**.

### 5a. A tool fails → does the agent recover?

In [ ]:
answer, _ = run_agent("What's on my calendar in calendar_november.csv?")
print("\n" + answer)

Look for the ❌ line. Did the model call `list_dir` next? Did it make up November events, or say the file doesn't exist?

**Try:** change `execute_tool` to `raise` instead of returning the error. What happens to the whole run?

### 5b. The model wants more turns than we allow

In [ ]:
answer, history = run_agent(TASK, max_turns=2)
print("\n" + answer)
print("messages in history:", len(history))

**Discuss:** Is returning `"[stopped after 2 turns]"` good enough? Alternatives:
- ask the model for a best-effort summary with tools disabled (one more call with no `tools`)
- warn the model *before* the budget runs out (Hermes does this, see Part 7)
- hand control back to a human

### 5c. Tool descriptions are prompts

The model chooses tools **only** from name + description + schema. Make them bad and watch.

In [ ]:
import copy

BAD_TOOLS = copy.deepcopy(TOOLS)
BAD_TOOLS["read_file"]["schema"]["description"] = "Gets things."
BAD_TOOLS["read_file"]["schema"]["input_schema"]["properties"]["path"]["description"] = "the thing"
BAD_TOOLS["list_dir"]["schema"]["description"] = "Misc."
# Rename read_file -> fetch (the dict key stays the lookup name, so rename both)
BAD_TOOLS["fetch"] = BAD_TOOLS.pop("read_file")
BAD_TOOLS["fetch"]["schema"]["name"] = "fetch"

answer, _ = run_agent("How many hours of open tasks do I have?", tools=BAD_TOOLS)
print("\n" + answer)

Compare the trace with the good tools. Count turns, look for wrong arguments (URLs? full paths?).

**And without the calculator?**

In [ ]:
NO_CALC = {k: v for k, v in TOOLS.items() if k != "calculator"}
answer, _ = run_agent("How many hours of open tasks do I have in total? Give the exact number.", tools=NO_CALC)
print("\n" + answer)

The exact answer is **19.75 h** (8 open tasks). Adding up decimals in its head: exact or not? Run it a few times.

## 6. Read the trace: where do the tokens go?

The API is **stateless**: every turn re-sends the *whole* history. Look at input tokens per turn.

In [ ]:
def token_report(task, **kw):
    usage = []
    _orig = trace_response
    def _record(turn, response):
        usage.append((turn, response.usage.input_tokens, response.usage.output_tokens))
        _orig(turn, response)
    globals()["trace_response"] = _record
    try:
        run_agent(task, **kw)
    finally:
        globals()["trace_response"] = _orig
    print("\nturn | input tokens | output tokens")
    for t, i, o in usage:
        print(f"{t:4} | {i:12,} | {o:13,}")
    print(f"total input: {sum(i for _, i, _ in usage):,}   total output: {sum(o for _, _, o in usage):,}")

token_report(TASK)

**Discuss:** Input tokens grow every turn. If each turn adds *k* tokens, the total cost after *n* turns grows like *n²*.
That is why real agents invest so much in **context management** (compaction, caching, clearing old tool results) — a later week.

## 7. Loops in the wild

Our loop is ~35 lines. Real agents keep exactly the same core and wrap a lot of engineering around it.
Read **`LOOPS_IN_THE_WILD.md`** (same folder) — it walks through the real loop code of **Hermes Agent** (Nous Research, Python) and **OpenClaw** (TypeScript, adapted from the pi agent).

| | Our loop | Hermes Agent | OpenClaw / pi |
|---|---|---|---|
| Core | `while turn < max_turns` | `while api_calls < max_iterations and budget.remaining > 0` | `while (hasMoreToolCalls \|\| pendingMessages.length > 0)` inside `while (true)` |
| Who stops it | model (`end_turn`) or turn cap | model, iteration budget, wall-clock budget, interrupt | model, abort signal, hooks (`shouldStopAfterTurn`), loop detector |
| Tool errors | returned as `is_error` | returned to model; invalid tool names answered with an error result | returned to model; truncated calls are failed, not run |
| Human mid-run | ✗ | interrupt + redirect | *steering* & *follow-up* message queues |
| Durability | none (in memory) | persists the tool call **before** executing it | event stream (`turn_start`, `message_end`, ...) |
| Lines | ~35 | ~1,800 in the loop module + per-phase modules | ~940 (pi) / ~1,450 (OpenClaw) |

**Discussion questions**
1. Strip Hermes and OpenClaw down to the lines that *must* be there. What's left? Is it our loop?
2. Each extra feature exists because something went wrong in production. Pick one and guess the incident.
3. Hermes appends a "budget almost used" warning to the newest tool result instead of the system prompt. Why there?
4. OpenClaw marks a turn as *tainted* once a tool returned content from the network. What attack is that preparing for?
5. Which of their features would you add to **our** loop first, and why?

## 8. Homework

### 8.1 From notebook to modules (required)

A notebook is a lab bench, not a product. Move your code into a package with this layout:

```
my_agent/
├── agent/
│   ├── __init__.py      # exports run_agent
│   ├── config.py        # MODEL, MAX_TOKENS, EFFORT, MAX_TURNS, DATA_DIR, SYSTEM_PROMPT
│   ├── tools.py         # tool functions, schemas, TOOLS registry, execute_tool
│   ├── tracing.py       # trace_response, trace_tool_result
│   ├── loop.py          # run_agent  (with Part 9: talks to the model only through providers.py)
│   ├── providers.py     # (if you did Part 9) adapters: Anthropic, OpenAI/Ollama Responses
│   └── router.py        # (if you did Part 9) classify, choose_model, run_routed
├── tests/
│   └── test_agent.py    # tests that run WITHOUT an API key
└── main.py              # python main.py "your task"
```

Rules:
- `loop.py` must not know which tools exist; it receives `tools` as a parameter.
- `run_agent` accepts a `client=` parameter so tests can pass a **fake client** that returns scripted responses.
- At least these tests: calculator rejects `__import__('os')`; `read_file('../x')` is blocked;
  the loop stops at `max_turns`; two tool calls in one reply → two results in **one** message.

### 8.2 A tool with side effects (required)

Add `write_file(path, content)`. Before it runs, print the path and the first lines of content and ask
`Allow? [y/N]` with `input()`. If the user says no, return an `is_error` result saying *the user declined* — don't crash.
Then ask the agent to *"write a plan for my week to week_plan.md"*.

### 8.3 Stretch
- Stop the run when total tokens exceed a budget you choose; report the cost in dollars.
- When `max_turns` is hit, make one last call **without tools** asking for a best-effort answer.
- Add a budget warning like Hermes: at 80% of `max_turns`, tell the model to wrap up.
- After Part 9: add **escalation**. Start on the routed model; if two tool errors happen in a row or 70% of
  `max_turns` is used, continue the *same task* on the next stronger model.

## 9. (Optional) Pick the model before the loop: routing across providers

Not every task needs the strongest model. Listing a folder doesn't need the same model as a multi-file analysis.
Idea: **classify the task with a cheap model first**, then run the loop with the model that fits.

```
task ──► cheap classifier ──► tier: simple | moderate | hard
                                   │
                ROUTES[tier] = [candidates in order of preference]
                                   │  (first one that is available)
                                   ▼
               provider + model ──► the same agentic loop
```

Two different kinds of control flow sit side by side here:
- **The router is a workflow.** Our code decides the steps: classify → look up → run.
- **The loop is an agent.** The model decides the steps.

We'll route across **three providers**: Claude (Anthropic), OpenAI, and a local model via **Ollama**.
Use whatever you have:

| Provider | What you need |
|---|---|
| Anthropic | `ANTHROPIC_API_KEY` (or `ant auth login`) |
| OpenAI | `OPENAI_API_KEY` |
| Ollama | Ollama ≥ 0.13.3 running locally, plus a tool-capable model: `ollama pull qwen3:8b` |

Providers you don't have are skipped automatically.

### 9.1 Same tool, different wire formats

Before we can switch providers, look at how differently they describe *the same* tool call:

|  | Anthropic Messages API | OpenAI Responses API (OpenAI **and** Ollama) |
|---|---|---|
| Tool definition | `{name, description, input_schema}` | `{type: "function", name, description, parameters}` |
| Model asks | content block `tool_use`: `id`, `name`, `input` (dict) | output item `function_call`: `call_id`, `name`, `arguments` (**JSON string**) |
| We answer | `user` message with `tool_result` blocks: `tool_use_id`, `content`, `is_error` | input item `function_call_output`: `call_id`, `output` (**no error flag**) |
| "Run my tools" | `stop_reason == "tool_use"` | any `function_call` in `output` |
| Cut off | `stop_reason == "max_tokens"` | `status == "incomplete"` |
| History | append `response.content` | append `response.output` (incl. reasoning items) |

The loop logic is the same; only the envelope changes. So we write an **adapter** per format and keep one loop.

In [ ]:
calc = TOOLS["calculator"]["schema"]
as_openai = {"type": "function", "name": calc["name"], "description": calc["description"],
             "parameters": calc["input_schema"]}
print("Anthropic:", json.dumps(calc, indent=1)[:300], "...\n")
print("OpenAI:   ", json.dumps(as_openai, indent=1)[:300], "...")

### 9.2 Neutral types + one adapter per format

The loop will only see three small types: `ToolCall`, `ToolResult` and `Step`.
Each adapter keeps the history **in its own native format** and translates at the edges.

In [ ]:
import os
import urllib.request
from dataclasses import dataclass, field

@dataclass
class ToolCall:
    id: str
    name: str
    input: dict

@dataclass
class ToolResult:
    tool_use_id: str
    content: str
    is_error: bool = False

@dataclass
class Step:
    stop: str                     # tool_use | end_turn | max_tokens | refusal
    text: str = ""
    tool_calls: list = field(default_factory=list)
    input_tokens: int = 0
    output_tokens: int = 0


CLAUDE_EFFORT_MODELS = {"claude-opus-5-5", "claude-sonnet-5-5"}   # Haiku 4.5 rejects effort/fallbacks

class AnthropicConversation:
    provider = "anthropic"

    def __init__(self, task, model, tools=(), system=SYSTEM_PROMPT, effort="medium"):
        self.client, self.model, self.system, self.effort = anthropic.Anthropic(), model, system, effort
        self.tools = list(tools)
        self.messages = [{"role": "user", "content": task}]

    def ask(self) -> Step:
        extra = {}
        if self.model in CLAUDE_EFFORT_MODELS:
            extra = {"output_config": {"effort": self.effort},
                     "betas": ["server-side-fallback-2026-07-01"], "fallbacks": "default"}
        r = self.client.beta.messages.create(model=self.model, max_tokens=16000, system=self.system,
                                             tools=self.tools, messages=self.messages, **extra)
        self.messages.append({"role": "assistant", "content": r.content})
        return Step(stop=str(r.stop_reason),
                    text="\n".join(b.text for b in r.content if b.type == "text").strip(),
                    tool_calls=[ToolCall(b.id, b.name, b.input) for b in r.content if b.type == "tool_use"],
                    input_tokens=r.usage.input_tokens, output_tokens=r.usage.output_tokens)

    def add_tool_results(self, results):
        self.messages.append({"role": "user", "content": [
            {"type": "tool_result", "tool_use_id": x.tool_use_id, "content": x.content,
             **({"is_error": True} if x.is_error else {})} for x in results]})

    def complete_json(self, schema) -> dict:
        r = self.client.messages.create(model=self.model, max_tokens=1024, system=self.system,
                                        messages=self.messages,
                                        output_config={"format": {"type": "json_schema", "schema": schema}})
        return json.loads(next(b.text for b in r.content if b.type == "text"))


OLLAMA_BASE_URL = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434/v1")

class ResponsesConversation:
    # OpenAI Responses API. Ollama serves the same API locally, so one adapter covers both.
    def __init__(self, task, model, provider="openai", tools=(), system=SYSTEM_PROMPT):
        from openai import OpenAI
        self.client = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama") if provider == "ollama" else OpenAI()
        self.model, self.provider, self.system = model, provider, system
        self.tools = [{"type": "function", "name": t["name"], "description": t["description"],
                       "parameters": t["input_schema"]} for t in tools]
        self.input = [{"role": "user", "content": task}]

    def ask(self) -> Step:
        r = self.client.responses.create(model=self.model, instructions=self.system, input=self.input,
                                         tools=self.tools, max_output_tokens=16000)
        self.input += r.output                              # everything back, incl. reasoning items
        calls = [ToolCall(i.call_id, i.name, json.loads(i.arguments or "{}"))
                 for i in r.output if i.type == "function_call"]
        refused = any(c.type == "refusal" for i in r.output if i.type == "message" for c in i.content)
        stop = ("max_tokens" if r.status == "incomplete" else "refusal" if refused
                else "tool_use" if calls else "end_turn")
        return Step(stop, (r.output_text or "").strip(), calls,
                    getattr(r.usage, "input_tokens", 0) or 0, getattr(r.usage, "output_tokens", 0) or 0)

    def add_tool_results(self, results):
        for x in results:   # no is_error field here, so say it in the text
            self.input.append({"type": "function_call_output", "call_id": x.tool_use_id,
                               "output": f"ERROR: {x.content}" if x.is_error else x.content})

    def complete_json(self, schema) -> dict:
        r = self.client.responses.create(
            model=self.model, instructions=self.system, input=self.input, max_output_tokens=1024,
            text={"format": {"type": "json_schema", "name": "result", "schema": schema, "strict": True}})
        return json.loads(r.output_text)


def open_conversation(provider, model, task, tools=(), system=SYSTEM_PROMPT):
    if provider == "anthropic":
        return AnthropicConversation(task, model, tools, system)
    return ResponsesConversation(task, model, provider, tools, system)


def ollama_models() -> set:
    try:
        with urllib.request.urlopen(OLLAMA_BASE_URL.removesuffix("/v1") + "/api/tags", timeout=0.5) as resp:
            return {m["name"] for m in json.load(resp).get("models", [])}
    except (OSError, ValueError):
        return set()

def is_available(provider, model) -> bool:
    if provider == "anthropic":
        return True
    if provider == "openai":
        return bool(os.environ.get("OPENAI_API_KEY"))
    if provider == "ollama":
        return model in ollama_models()
    return False

### 9.3 One loop for every provider

Compare with `run_agent` from Part 4. What changed?

In [ ]:
def run_tool(call, tools=TOOLS) -> ToolResult:
    r = execute_tool(call, tools)          # Part 3's function, unchanged
    return ToolResult(r["tool_use_id"], str(r["content"]), r.get("is_error", False))

def run_agent_any(conv, tools=TOOLS, max_turns=10, verbose=True):
    total_in = total_out = 0
    for turn in range(1, max_turns + 1):
        step = conv.ask()
        total_in, total_out = total_in + step.input_tokens, total_out + step.output_tokens
        if verbose:
            print(f"\n── turn {turn} · {conv.provider}/{conv.model} · stop={step.stop}")
            if step.text:
                print("  💬", _short(step.text, 300))
            for c in step.tool_calls:
                print(f"  🔧 {c.name}({c.input})")
        if step.stop == "tool_use":
            results = [run_tool(c, tools) for c in step.tool_calls]
            if verbose:
                for r in results:
                    print("  ❌" if r.is_error else "  ✅", _short(r.content))
            conv.add_tool_results(results)
            continue
        if step.stop == "max_tokens":
            return step.text + "\n[cut off: max_tokens]", turn, total_in, total_out
        if step.stop == "refusal":
            return "[the model declined this request]", turn, total_in, total_out
        return step.text, turn, total_in, total_out
    return f"[stopped after {max_turns} turns]", max_turns, total_in, total_out

schemas = [t["schema"] for t in TOOLS.values()]
answer, turns, tin, tout = run_agent_any(open_conversation("anthropic", "claude-sonnet-5-5",
                                                           "How many hours of open tasks do I have?", schemas))
print(f"\n{answer}\n[{turns} turns, {tin} in / {tout} out]")

### 9.4 The classifier

A cheap, fast model reads the task and returns a **tier**. Two rules:

1. **Constrain the output.** We use structured outputs (a JSON schema with an `enum`), so the answer is always parseable.
2. **Never trust it blindly.** If the call fails or returns something unexpected, fall back to the **strongest** tier.
   A hard task on a weak model is a worse failure than an easy task on a strong one.

In [ ]:
TIERS = ("simple", "moderate", "hard")
DEFAULT_TIER = "hard"
CLASSIFIER = ("anthropic", "claude-haiku-4-5")      # try ("ollama", "qwen3:8b") or ("openai", "gpt-6-luna")

CLASSIFIER_PROMPT = '''You route tasks for a personal assistant. The assistant has tools to list and read files
(calendar, tasks, notes) and a calculator. Classify how demanding the task is FOR THAT AGENT:

- simple:   one or two tool calls, no real reasoning (list files, read one file, one sum)
- moderate: several tool calls and straightforward arithmetic over one or two files
- hard:     multi-step reasoning across several files, time arithmetic, judgement or ambiguity

Answer with the tier and a one-sentence reason.'''

CLASSIFIER_SCHEMA = {
    "type": "object",
    "properties": {"tier": {"type": "string", "enum": list(TIERS)}, "reason": {"type": "string"}},
    "required": ["tier", "reason"],
    "additionalProperties": False,
}

def classify(task):
    provider, model = CLASSIFIER
    try:
        answer = open_conversation(provider, model, task, system=CLASSIFIER_PROMPT).complete_json(CLASSIFIER_SCHEMA)
        if answer.get("tier") in TIERS:
            return answer["tier"], answer.get("reason", "")
        return DEFAULT_TIER, f"unknown tier {answer.get('tier')!r}"
    except Exception as exc:
        return DEFAULT_TIER, f"classifier failed ({type(exc).__name__}: {exc})"

TASKS = [
    "List the files in the data folder.",
    "How many hours of open tasks do I have?",
    TASK,   # the multi-file growth analysis from Part 4
]
for t in TASKS:
    print(classify(t), "←", t[:60])

### 9.5 The route table

For each tier, a list of candidates **in order of preference**. The router takes the first one that's available.
If a whole tier is unavailable it tries **stronger** tiers first, and weaker ones only as a last resort.

> Model names change often. Check each provider's model list and edit the table. It's data, not code.

In [ ]:
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "qwen3:8b")

ROUTES = {
    "simple":   [("ollama", OLLAMA_MODEL), ("openai", "gpt-6-luna"), ("anthropic", "claude-haiku-4-5")],
    "moderate": [("openai", "gpt-6.1-sol"), ("anthropic", "claude-sonnet-5-5")],
    "hard":     [("anthropic", "claude-opus-5-5"), ("openai", "gpt-6-astra")],
}

def choose_model(tier):
    i = TIERS.index(tier)
    for t in TIERS[i:] + TIERS[:i][::-1]:            # this tier, stronger tiers, then weaker
        for provider, model in ROUTES[t]:
            if is_available(provider, model):
                return provider, model
    raise RuntimeError("No configured model is available")

for tier, candidates in ROUTES.items():
    print(f"{tier:9}", "  ".join(f"{'✓' if is_available(p, m) else '✗'} {p}/{m}" for p, m in candidates))

### 9.6 Route, then loop

In [ ]:
def run_routed(task, verbose=True):
    tier, reason = classify(task)
    provider, model = choose_model(tier)
    print(f"\n🧭 tier={tier} → {provider}/{model}   ({reason})")
    conv = open_conversation(provider, model, task, [t["schema"] for t in TOOLS.values()])
    answer, turns, tin, tout = run_agent_any(conv, verbose=verbose)
    return {"task": task[:40], "tier": tier, "model": f"{provider}/{model}",
            "turns": turns, "in": tin, "out": tout, "answer": answer}

routed = [run_routed(t, verbose=False) for t in TASKS]
for r in routed:
    print(f"\n{r['tier']:9} {r['model']:32} {r['turns']} turns  {r['in']:>6} in {r['out']:>5} out\n   {_short(r['answer'], 160)}")

### 9.7 Experiment: is routing worth it?

Routing only pays off if it saves more than it costs. Run the same tasks three ways and fill in the table:

| Setup | What it is |
|---|---|
| **A. Always strongest** | Opus 5.5, effort `medium` for every task |
| **B. Routed** | classifier + route table (above) |
| **C. One model, effort tuned** | Opus 5.5, effort `low` for every task (no classifier, one model) |

Prices are per million tokens. Fill in the ones marked `None` from the provider's pricing page. Local models cost 0 (but not *nothing*: your hardware and time).

In [ ]:
PRICES = {   # (input $, output $) per 1M tokens
    "anthropic/claude-haiku-4-5": (1.00, 5.00),
    "anthropic/claude-sonnet-5-5": (2.00, 10.00),
    "anthropic/claude-opus-5-5": (4.00, 20.00),
    "openai/gpt-6-luna": (0.10, 0.50),
    "openai/gpt-6.1-sol": None,
    "openai/gpt-6-astra": None,
    f"ollama/{OLLAMA_MODEL}": (0.0, 0.0),
}

def cost(model, tin, tout):
    p = PRICES.get(model)
    return None if p is None else (tin * p[0] + tout * p[1]) / 1e6

def run_fixed(task, model="claude-opus-5-5", effort="medium"):
    conv = AnthropicConversation(task, model, [t["schema"] for t in TOOLS.values()], effort=effort)
    answer, turns, tin, tout = run_agent_any(conv, verbose=False)
    return {"model": f"anthropic/{model}", "turns": turns, "in": tin, "out": tout, "answer": answer}

rows = []
for i, t in enumerate(TASKS):
    a, c = run_fixed(t, effort="medium"), run_fixed(t, effort="low")
    b = routed[i]
    for name, r in (("A strongest", a), ("B routed", b), ("C opus low", c)):
        rows.append((i + 1, name, r["model"], r["turns"], cost(r["model"], r["in"], r["out"]), r["answer"]))

print("task | setup        | model                          | turns | cost $")
for task_no, name, model, turns, usd, _ in rows:
    print(f"{task_no:4} | {name:12} | {model:30} | {turns:5} | {'?' if usd is None else f'{usd:.4f}'}")

Now check **correctness** against the ground truth from Part 4. Cheap but wrong is not cheap.

```python
for task_no, name, model, *_, answer in rows: print(task_no, name, "→", _short(answer, 200))
```

**Discuss:**
1. Add the **classifier's** cost and latency to setup B. On which tasks does routing still win?
2. Did any task get misrouted? What did it cost: a wrong answer, or more turns?
3. Setup C has no classifier and one model. How close does it get to B? (The prompt cache only works
   within one model, so switching models loses it. One model + effort tuning is often the simpler win.)
4. The local model: did it call the tools correctly? Small models often struggle with tool calling, so the "cheap" tier can need more turns.
5. **Routing vs escalation.** We choose once, *before* the loop. OpenClaw's `prepareNextTurn` hook can switch models
   *between turns* (see `LOOPS_IN_THE_WILD.md`). When would "start cheap, escalate if stuck" beat classifying up front?
6. Hermes supports many providers by converting everything into one internal message format.
   We kept each provider's native format inside its adapter instead. What are the trade-offs?